# Notebook A — Labels, Dual-Task Leakage Audit, and Frozen Splits

The paper's central move happens here: **the same diagnostic protocol is run on two tasks
built from one corpus, and it returns opposite verdicts.**

| | Task A | Task B |
|---|---|---|
| Label | `continent` (derived from GPS) | `scene` (CLIP teacher) |
| Metadata under test | GPS and its derivatives | GPS and its derivatives |
| Expected verdict | **FAIL** — metadata is the target | **PASS** — metadata is informative, not determinative |

Task A costs nothing: it needs no images and runs on CPU in minutes. Task B is the study.
Together they demonstrate the protocol discriminates rather than always crying leakage —
which is the first objection a sharp reviewer will raise.

Everything the paper or repo needs is written to `paper_assets/` with an index.

In [ ]:
import json, math, warnings, hashlib, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import BallTree

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='paper', font_scale=1.05)
plt.rcParams['savefig.dpi'] = 300
np.random.seed(0)

EXT_DIR  = Path('/kaggle/input/nb0-extraction')       # Notebook 0 output
META_CSV = Path('/kaggle/input/yfcc100m-metadata/metadata.csv')  # YOUR notebook-1 output
OUT      = Path('/kaggle/working'); OUT.mkdir(exist_ok=True, parents=True)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  ARTIFACT REGISTRY — every figure/table the paper cites goes through this
# ════════════════════════════════════════════════════════════════════════
class Artifacts:
    """Writes figures as PNG(300dpi)+PDF and tables as CSV+LaTeX into
    paper_assets/, and maintains index.json + README.md for the GitHub repo."""
    def __init__(self, root):
        self.root = Path(root)
        (self.root / 'figures').mkdir(parents=True, exist_ok=True)
        (self.root / 'tables').mkdir(parents=True, exist_ok=True)
        self.idx_path = self.root / 'index.json'
        self.items = json.loads(self.idx_path.read_text()) if self.idx_path.exists() else []

    def _put(self, rec):
        self.items = [i for i in self.items if i['name'] != rec['name']] + [rec]
        self.idx_path.write_text(json.dumps(self.items, indent=2))

    def figure(self, fig, name, caption, section):
        for ext in ('png', 'pdf'):
            fig.savefig(self.root / 'figures' / f'{name}.{ext}', bbox_inches='tight')
        self._put({'kind': 'figure', 'name': name, 'caption': caption,
                   'section': section, 'files': [f'figures/{name}.png', f'figures/{name}.pdf']})
        print(f'  [artifact] figure {name}  -> section {section}')

    def table(self, df, name, caption, section, float_fmt='%.4f'):
        df.to_csv(self.root / 'tables' / f'{name}.csv', index=False)
        (self.root / 'tables' / f'{name}.tex').write_text(
            df.to_latex(index=False, escape=False, float_format=float_fmt,
                        caption=caption, label=f'tab:{name}'))
        self._put({'kind': 'table', 'name': name, 'caption': caption,
                   'section': section, 'files': [f'tables/{name}.csv', f'tables/{name}.tex']})
        print(f'  [artifact] table  {name}  -> section {section}')

    def readme(self):
        lines = ['# Paper assets', '',
                 'Auto-generated. Every figure and table cited in the paper, with its section.', '',
                 '| Section | Kind | Name | Caption |', '|---|---|---|---|']
        for i in sorted(self.items, key=lambda r: (str(r['section']), r['name'])):
            lines.append(f"| {i['section']} | {i['kind']} | `{i['name']}` | {i['caption'][:110]} |")
        (self.root / 'README.md').write_text('\n'.join(lines))
        print(f'\n{len(self.items)} artifacts indexed in {self.root}')

ART = Artifacts(OUT / 'paper_assets')

In [ ]:
# ── Load extraction output ──────────────────────────────────────────────
tab  = pd.read_parquet(EXT_DIR / 'extract_table.parquet')
LOG  = np.load(EXT_DIR / 'clip_logits.npy').astype(np.float32)
EMB  = np.load(EXT_DIR / 'resnet_emb.npy')
EX   = json.loads((EXT_DIR / 'extract_meta.json').read_text())
SHORT = EX['short']
print(f'{len(tab):,} images | clip {LOG.shape} | resnet {EMB.shape}')
assert len(tab) == len(LOG) == len(EMB)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  TASK B LABELS — CLIP teacher with confidence + margin filtering
# ════════════════════════════════════════════════════════════════════════
CONF_MIN, MARGIN_MIN, PER_CLASS_CAP = 0.30, 0.10, 40_000

prob = np.exp(LOG - LOG.max(1, keepdims=True))
prob /= prob.sum(1, keepdims=True)
top2 = np.sort(prob, axis=1)[:, -2:]
tab['scene_idx']  = prob.argmax(1)
tab['scene_conf'] = top2[:, 1]
tab['scene_marg'] = top2[:, 1] - top2[:, 0]
tab['scene']      = [SHORT[i] for i in tab.scene_idx]

sel = (tab.scene_conf >= CONF_MIN) & (tab.scene_marg >= MARGIN_MIN)
print(f'confidence/margin filter keeps {sel.mean():.1%} ({sel.sum():,} images)')

rng = np.random.RandomState(0)
keep_idx = []
for c in range(len(SHORT)):
    cand = np.where(sel.values & (tab.scene_idx.values == c))[0]
    if len(cand) > PER_CLASS_CAP:
        cand = rng.choice(cand, PER_CLASS_CAP, replace=False)
    keep_idx.append(cand)
keep_idx = np.sort(np.concatenate(keep_idx))

df   = tab.iloc[keep_idx].reset_index(drop=True)
LOG  = LOG[keep_idx]; EMB = EMB[keep_idx]
print(f'\nTask B corpus: {len(df):,} images')
display(df.scene.value_counts().to_frame('n').assign(pct=lambda d: (d.n/len(df)*100).round(1)))

In [ ]:
# ── Label-noise verification sample (hand-check these, report the rate) ─
V = 200
vs = df.sample(V, random_state=7)[['id', 'scene', 'scene_conf', 'scene_marg']].copy()
vs['human_label'] = ''      # <- fill this column in by hand
vs['correct'] = ''
vs.to_csv(OUT / 'label_verification_sample.csv', index=False)
print(f'Wrote label_verification_sample.csv ({V} rows).')
print('Fill `human_label`, compute the agreement rate, and report it in Section 3.2.')
print('Teacher labels are noisy; your claims are deltas measured on identical labels, '
      'but you must state the noise rate for that argument to hold.')

fig, axes = plt.subplots(3, 4, figsize=(13, 9))
for ax, c in zip(axes.ravel(), SHORT):
    s = df[df.scene == c].scene_conf
    ax.hist(s, bins=30, color='#2980b9')
    ax.set_title(f'{c}  (n={len(s):,}, med conf {s.median():.2f})', fontsize=9)
    ax.set_yticks([])
fig.suptitle('Teacher confidence by predicted scene class', fontweight='bold')
plt.tight_layout()
ART.figure(fig, 'figA1_teacher_confidence',
           'Distribution of CLIP teacher confidence per assigned scene class.', '3.2')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  TASK A LABELS — continent, joined from your Notebook-1 metadata
# ════════════════════════════════════════════════════════════════════════
try:
    m1 = pd.read_csv(META_CSV, usecols=lambda c: c in
                     {'id', 'country_code', 'continent', 'climate_zone', 'hemisphere'})
    df = df.merge(m1, on='id', how='left')
    cov = df.continent.notna().mean()
    print(f'joined notebook-1 metadata | continent coverage {cov:.1%}')
except Exception as e:
    print('Could not join notebook-1 metadata:', e)
    cov = 0.0

if cov < 0.5:
    print('Falling back to offline reverse geocoding (pip install reverse_geocoder).')
    import reverse_geocoder as rg
    import pycountry_convert as pc
    hits = rg.search(list(zip(df.lat.values, df.lon.values)))
    df['country_code'] = [h['cc'] for h in hits]
    def to_cont(cc):
        try:
            return pc.convert_continent_code_to_continent_name(
                pc.country_alpha2_to_continent_code(cc))
        except Exception:
            return np.nan
    df['continent'] = df.country_code.map(to_cont)

df['hemisphere']   = np.where(df.lat >= 0, 'N', 'S')
df['climate_zone'] = pd.cut(df.lat.abs(), [-1, 23.5, 35, 55, 66.5, 91],
                            labels=['tropical', 'subtropical', 'temperate',
                                    'subpolar', 'polar']).astype(str)
print(f'\nTask A: {df.continent.notna().sum():,} labelled')
display(df.continent.value_counts())

In [ ]:
# ── Encode both targets ─────────────────────────────────────────────────
MIN_PER_CLASS = 500
leB = LabelEncoder(); df['yB'] = leB.fit_transform(df.scene)

dA = df[df.continent.notna() & ~df.continent.isin(['Other'])]
okA = dA.continent.value_counts()[lambda s: s >= MIN_PER_CLASS].index
maskA = df.continent.isin(okA)
leA = LabelEncoder().fit(sorted(okA))
df['yA'] = np.where(maskA, leA.transform(df.continent.where(maskA, okA[0])), -1)

print(f'Task A: {(df.yA >= 0).sum():,} rows, {len(leA.classes_)} classes {list(leA.classes_)}')
print(f'Task B: {len(df):,} rows, {len(leB.classes_)} classes')
MAJ_A = df[df.yA >= 0].continent.value_counts(normalize=True).max()
MAJ_B = df.scene.value_counts(normalize=True).max()
print(f'majority baselines — A {MAJ_A:.4f} | B {MAJ_B:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  THE DIAGNOSTIC PROTOCOL — run identically on both tasks
# ════════════════════════════════════════════════════════════════════════
def cond_entropy(y, m):
    d = pd.DataFrame({'y': y, 'm': m}); tot, H = len(d), 0.0
    for _, g in d.groupby('m'):
        p = g.y.value_counts(normalize=True).values
        H += (len(g)/tot) * (-(p*np.log2(p)).sum())
    return H

def entropy(y):
    p = pd.Series(y).value_counts(normalize=True).values
    return -(p*np.log2(p)).sum()

def meta_only_score(X, y, seed=0):
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=150, random_state=seed),
                          X, y, cv=cv)
    return accuracy_score(y, p), f1_score(y, p, average='macro')

GROUPS = {
    'gps_exact'  : lambda d: d[['lat', 'lon']].values,
    'gps_1deg'   : lambda d: np.floor(d[['lat', 'lon']].values / 1.0),
    'gps_5deg'   : lambda d: np.floor(d[['lat', 'lon']].values / 5.0),
    'country'    : lambda d: pd.Categorical(d.country_code).codes.reshape(-1, 1),
    'climate'    : lambda d: pd.Categorical(d.climate_zone).codes.reshape(-1, 1),
    'hemisphere' : lambda d: pd.Categorical(d.hemisphere).codes.reshape(-1, 1),
    'file_meta'  : lambda d: d[['aspect', 'megapix', 'bytes_px']].values,
}

rows = []
for task, ycol, maj in [('A: continent', 'yA', MAJ_A), ('B: scene', 'yB', MAJ_B)]:
    sub = df[df[ycol] >= 0]
    ss  = sub.sample(min(60_000, len(sub)), random_state=0)   # audit on a subsample; it is a diagnostic
    y   = ss[ycol].values
    Hy  = entropy(y)
    for g, fn in GROUPS.items():
        X = fn(ss)
        acc, f1 = meta_only_score(X, y)
        key = pd.Series([tuple(r) for r in np.atleast_2d(X)])
        # H(y|m) is only meaningful for discrete metadata. With continuous values every
        # row is its own bin and H(y|m) is trivially 0 -- which would falsely read as
        # deterministic leakage. Report NaN there and rely on metadata-only accuracy.
        discrete = key.nunique() <= len(key) / 10
        rows.append({'task': task, 'metadata': g, 'H(y)': Hy,
                     'H(y|m)': cond_entropy(y, key) if discrete else np.nan,
                     'n_distinct': int(key.nunique()), 'discrete': bool(discrete),
                     'meta_only_acc': acc, 'meta_only_f1': f1,
                     'majority': maj, 'lift': acc - maj})

audit = pd.DataFrame(rows)
audit['info_captured_%'] = 100 * (1 - audit['H(y|m)'] / audit['H(y)'])
audit['verdict'] = np.where(audit.meta_only_acc > 0.95, 'FAIL: target in disguise',
                    np.where(audit.lift > 0.25, 'CAUTION: dominates image',
                    np.where(audit.lift > 0.02, 'PASS: informative', 'PASS: negligible')))
display(audit.round(4))
ART.table(audit.round(4), 'tab1_dual_task_audit',
          'Leakage audit applied identically to both tasks. Metadata-only accuracy and '
          'conditional entropy show the same GPS metadata is target-determining for Task A '
          'and merely informative for Task B.', '4.1')

In [ ]:
# ── FIGURE: the paper's headline diagnostic ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharey=False)
for ax, (task, maj) in zip(axes, [('A: continent', MAJ_A), ('B: scene', MAJ_B)]):
    s = audit[audit.task == task].sort_values('meta_only_acc')
    cols = ['#c0392b' if a > 0.95 else '#e67e22' if a - maj > 0.25 else '#2980b9'
            for a in s.meta_only_acc]
    ax.barh(s.metadata, s.meta_only_acc, color=cols)
    ax.axvline(maj, ls='--', c='k', lw=1.4, label=f'majority = {maj:.2f}')
    ax.axvline(0.95, ls=':', c='#c0392b', lw=1.4, label='leakage threshold')
    ax.set_xlim(0, 1); ax.set_xlabel('accuracy from metadata alone')
    ax.set_title(f'Task {task}', fontweight='bold'); ax.legend(fontsize=8, loc='lower right')
fig.suptitle('One protocol, two verdicts: identical metadata is the target in Task A '
             'and a genuine prior in Task B', fontweight='bold', fontsize=11.5)
plt.tight_layout()
ART.figure(fig, 'fig1_dual_task_audit',
           'Metadata-only accuracy for each metadata group under both tasks.', '4.1')
plt.show()

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  SPATIAL AUTOCORRELATION + SPLIT CONSTRUCTION
# ════════════════════════════════════════════════════════════════════════
BLOCK_DEG, VAL_F, TEST_F, SPLIT_SEED = 5.0, 0.15, 0.15, 42
EARTH = 6371.0
coords = np.radians(df[['lat', 'lon']].values)

def blocked_split(keys, y, val_f, test_f, seed):
    rs = np.random.RandomState(seed)
    k = pd.DataFrame({'k': keys, 'y': y})
    dom = k.groupby('k').y.agg(lambda s: s.value_counts().idxmax())
    sz  = k.groupby('k').size()
    assign = {}
    for cls in sorted(k.y.unique()):
        gs = dom[dom == cls].index.to_numpy(); rs.shuffle(gs)
        gs = gs[np.argsort(-sz[gs].values, kind='stable')]
        tot = sz[gs].sum()
        tgt = {'train': (1-val_f-test_f)*tot, 'val': val_f*tot, 'test': test_f*tot}
        cur = {'train': 0.0, 'val': 0.0, 'test': 0.0}
        for g in gs:
            f = max(tgt, key=lambda z: tgt[z] - cur[z]); assign[g] = f; cur[f] += sz[g]
    fo = np.array([assign[x] for x in keys])
    return {s: np.where(fo == s)[0] for s in ('train', 'val', 'test')}

def random_split(y, val_f, test_f, seed):
    i = np.arange(len(y))
    trv, te = train_test_split(i, test_size=test_f, stratify=y, random_state=seed)
    tr, va = train_test_split(trv, test_size=val_f/(1-test_f), stratify=y[trv], random_state=seed)
    return {'train': tr, 'val': va, 'test': te}

bkey = (np.floor(df.lat/BLOCK_DEG).astype(int).astype(str) + '_' +
        np.floor(df.lon/BLOCK_DEG).astype(int).astype(str)).values
SPL = {'random': random_split(df.yB.values, VAL_F, TEST_F, SPLIT_SEED),
       'spatial': blocked_split(bkey, df.yB.values, VAL_F, TEST_F, SPLIT_SEED)}

sep = []
for nm, s in SPL.items():
    t = BallTree(coords[s['train']], metric='haversine')
    d = t.query(coords[s['test']], k=1)[0].ravel() * EARTH
    sep.append({'protocol': nm, 'train_%': len(s['train'])/len(df)*100,
                'test_%': len(s['test'])/len(df)*100,
                'median_km': np.median(d), 'within_100m_%': (d < 0.1).mean()*100,
                'within_1km_%': (d < 1).mean()*100})
    print(f"{nm:8s} train={len(s['train']):,} val={len(s['val']):,} test={len(s['test']):,} "
          f"| median test->train {np.median(d):.2f} km | within 1 km {(d<1).mean()*100:.1f}%")
sepdf = pd.DataFrame(sep)
ART.table(sepdf.round(3), 'tab2_split_separation',
          'Spatial separation achieved by each split protocol.', '3.5')

fig, ax = plt.subplots(figsize=(7.5, 4.2))
for nm, c in zip(SPL, ['#c0392b', '#2980b9']):
    t = BallTree(coords[SPL[nm]['train']], metric='haversine')
    d = np.clip(t.query(coords[SPL[nm]['test']], k=1)[0].ravel()*EARTH, 1e-3, None)
    ax.hist(np.log10(d), bins=60, alpha=0.55, label=nm, color=c, density=True)
ax.set_xlabel('log10 distance from test image to nearest training image (km)')
ax.set_ylabel('density'); ax.legend()
ax.set_title('Random splitting places near-duplicates on both sides', fontweight='bold')
plt.tight_layout()
ART.figure(fig, 'fig2_split_separation',
           'Test-to-train nearest-neighbour distance under each split protocol.', '3.5')
plt.show()

In [ ]:
# ── Freeze everything ───────────────────────────────────────────────────
np.save(OUT / 'resnet_emb.npy', EMB)
df.to_parquet(OUT / 'corpus.parquet', index=False)
json.dump({'classesA': leA.classes_.tolist(), 'classesB': leB.classes_.tolist(),
           'majorityA': float(MAJ_A), 'majorityB': float(MAJ_B),
           'block_deg': BLOCK_DEG, 'seed': SPLIT_SEED, 'n': int(len(df)),
           'conf_min': CONF_MIN, 'margin_min': MARGIN_MIN, 'cap': PER_CLASS_CAP,
           'splits': {k: {p: v.tolist() for p, v in s.items()} for k, s in SPL.items()}},
          open(OUT / 'splits.json', 'w'))
(OUT / 'data_hash.txt').write_text(
    hashlib.md5(pd.util.hash_pandas_object(df[['id', 'yA', 'yB']]).values.tobytes()).hexdigest())
ART.readme()
print('\nSaved corpus.parquet, resnet_emb.npy, splits.json, paper_assets/')


---
**Gate:** Task B's audit must show GPS well above the majority baseline but well below 0.95.
If `gps_exact` exceeds 0.95 on Task B, your scene labels are geographically degenerate —
re-check the taxonomy before spending GPU time on Notebook C.

**Next:** publish this notebook's output as a Dataset, attach to Notebook C.